In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
file_path = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/refs/heads/main/data/2020/2020-01-21/spotify_songs.csv"

df = pd.read_csv(file_path)
df.head()

,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,66,2oCs0DGTsRO98Gh5ZSl2Cx,I Don't Care (with Justin Bieber) [Loud Luxury...,2019-06-14,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,6,-2.634,1,0.0583,0.1020,0.000000,0.0653,0.518,122.036,194754
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,63rPSO264uRjW1X5E6cWv6,Memories (Dillon Francis Remix),2019-12-13,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,11,-4.969,1,0.0373,0.0724,0.004210,0.3570,0.693,99.972,162600
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,1HoSmj2eLcsrR0vE9gThr4,All the Time (Don Diablo Remix),2019-07-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-3.432,0,0.0742,0.0794,0.000023,0.1100,0.613,124.008,176616
3,75FpbthrwQmzHlBJLuGdC7,Call You Mine - Keanu Silva Remix,The Chainsmokers,60,1nqYsOef1yKKuGOVchbsk6,Call You Mine - The Remixes,2019-07-19,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,7,-3.778,1,0.1020,0.0287,0.000009,0.2040,0.277,121.956,169093
4,1e8PAfcKUYoKkxPhrHqw4x,Someone You Loved - Future Humans Remix,Lewis Capaldi,69,7m7vv9wlQ4i0LFuJiE2zsQ,Someone You Loved (Future Humans Remix),2019-03-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-4.672,1,0.0359,0.0803,0.000000,0.0833,0.725,123.976,189052


In [4]:
print("original dataset's shape" , df.shape)

original dataset's shape (32833, 23)


## Data cleaning

In [5]:
#####
# time processing

# Duration in minutes is easier to read than raw milliseconds
df['duration_min'] = df['duration_ms'] / 60000
df["duration_min"].describe()

,duration_min
count,32833.000000
mean,3.763330
std,0.997233
min,0.066667
25%,3.130317
50%,3.600000
75%,4.226417
max,8.630167


In [6]:
#####
# year processing

s = df["track_album_release_date"].astype(str).str.strip()

# detect year-only values like "2020"
year_only = s.str.fullmatch(r"\d{4}")

# prepare output column (nullable integer)
df["release_year"] = pd.Series(pd.NA, index=df.index, dtype="Int64")

# 1️⃣ year-only values → use directly
df.loc[year_only, "release_year"] = s[year_only].astype(int)

# 2️⃣ all other date formats → parse then extract year
parsed = pd.to_datetime(
    s[~year_only],
    dayfirst=False,
    errors="coerce"
)

df.loc[~year_only, "release_year"] = parsed.dt.year
df["release_year"].describe()

,release_year
count,32802.0
mean,2011.173587
std,11.360195
min,1957.0
25%,2008.0
50%,2016.0
75%,2019.0
max,2020.0


In [7]:
df.isna().sum()[df.isna().sum() > 0]

,0
track_name,5
track_artist,5
track_album_name,5
release_year,31


In [8]:
df = df.dropna()

In [9]:
df.isna().sum()[df.isna().sum() > 0]

,0


In [10]:
df = df[df["release_year"] >= 1970]

In [11]:
print("cleaned dataset's shape" , df.shape)
print("=" * 55)
print(df.describe())

cleaned dataset's shape (32631, 25)
       track_popularity  danceability        energy           key  \
count      32631.000000  32631.000000  32631.000000  32631.000000   
mean          42.448745      0.655681      0.699093      5.378842   
std           24.975076      0.144711      0.180651      3.611898   
min            0.000000      0.000000      0.000175      0.000000   
25%           24.000000      0.564000      0.582000      2.000000   
50%           45.000000      0.672000      0.722000      6.000000   
75%           62.000000      0.761000      0.840000      9.000000   
max          100.000000      0.983000      1.000000     11.000000   

           loudness          mode   speechiness  acousticness  \
count  32631.000000  32631.000000  32631.000000  32631.000000   
mean      -6.701845      0.564525      0.107368      0.174487   
std        2.979180      0.495827      0.101466      0.219107   
min      -46.448000      0.000000      0.000000      0.000000   
25%       -8.1450

In [12]:
df.to_csv('cleaned_spotify.csv', index=False)